<a href="https://colab.research.google.com/github/Kareena-3/FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Data Contract

## CTR / Engagement Opportunity Scoring

**Lane:** CTR / Engagement Opportunity Scoring

**Decision:** Which content pages should a reviewer inspect first because their observed CTR looks weak relative to similar search positions?

**Unit of analysis:** One row in the final feature frame = one content page for one client in one calendar month.

**Table:** `fact_content_daily_performance` from the FlyRank internship warehouse.

**Development window:** March 2026. I use a mid-panel month rather than the final June 2026 `_sample` month, because June is reserved as a sealed outcome/test month.

**What I rank:** A position-adjusted CTR opportunity proxy.

**Deliberately excluded:** outcome-derived fields such as `trend_direction` and `trend_pct`, because they would leak the outcome into the features.

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass
import pandas as pd
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Enter your Hugging Face READ token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
MONTH = '2026-03'
print('Warehouse connected. Development month:', MONTH)

Warehouse connected. Development month: 2026-03


## 1–2. Contract in plain words

The raw fact is daily. For my lane I aggregate it to page-month. The decision is which pages deserve CTR/content review first.

**Proxy:** `expected_ctr` is the average CTR for the same position band; `ctr_gap = observed CTR - expected CTR`; `ctr_opportunity_proxy = -ctr_gap`. This is an observed proxy, not proof that changing a page will cause a result.

## 3. Exactly three verification queries

### Query 1 — grain
Confirm there is no duplicate at `report_date × client_hash_id × content_hash_id`.

In [2]:
grain_check = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM {FACT}
WHERE month = '{MONTH}'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
""").df()
print('Duplicate groups at the documented daily grain:')
display(grain_check)
print('Duplicate groups found:', len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate groups at the documented daily grain:


,report_date,client_hash_id,content_hash_id,n


Duplicate groups found: 0


### Query 2 — row count and date span

In [3]:
slice_check = con.sql(f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS first_date, MAX(report_date) AS last_date
FROM {FACT}
WHERE month = '{MONTH}'
""").df()
display(slice_check)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — availability
Use `IS TRUE` so NULL is not treated as available.

In [4]:
availability_check = con.sql(f"""
SELECT COUNT(*) AS rows_with_both_sources
FROM {FACT}
WHERE month = '{MONTH}'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
""").df()
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_both_sources
0,364347


## 3. Five features

1. **impressions** — knowable at decision time because Search Console exposure has already been observed.
2. **clicks** — knowable because Search Console clicks have already been observed.
3. **avg_position** — knowable because search position is already observed.
4. **sessions** — knowable when Analytics data is available.
5. **engaged_sessions** — knowable when Analytics data is available.

IDs are context only, not model features.

In [5]:
features = con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS impressions,
       SUM(gsc_clicks) AS clicks,
       AVG(NULLIF(gsc_avg_position, 0)) AS avg_position,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_sessions
FROM {FACT}
WHERE month = '{MONTH}'
GROUP BY 1,2
""").df()
features['ctr'] = features['clicks'] / features['impressions'].replace(0, pd.NA)
print('One row = one page-month. Rows:', len(features))
display(features.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

One row = one page-month. Rows: 331437


,client_hash_id,content_hash_id,impressions,clicks,avg_position,sessions,engaged_sessions,ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0,0.001073
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.307255,0.0,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,0.0,0.001066
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,0.0,0.002629
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,23.314103,7.0,0.0,0.0
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.499519,2.0,0.0,0.002331
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,9.760489,2.0,0.0,0.004484
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,8.475000,8.0,0.0,0.010417
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,9.155335,0.0,0.0,0.003185
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,5.258331,12.0,0.0,0.002594


### Target / proxy sketch

A larger positive `ctr_opportunity_proxy` means a page looks weaker than its position peers. This is only an observed association. A future supervised model should ideally use a later outcome window.

In [6]:
features['position_band'] = pd.cut(
    features['avg_position'],
    bins=[-float('inf'),3,10,20,50,float('inf')],
    labels=['top_3','page_1','striking','page_3_5','deep']
)
features['expected_ctr'] = features.groupby('position_band', observed=False)['ctr'].transform('mean')
features['ctr_gap'] = features['ctr'] - features['expected_ctr']
features['ctr_opportunity_proxy'] = -features['ctr_gap']
display(features[['client_hash_id','content_hash_id','position_band','impressions','clicks','ctr','expected_ctr','ctr_gap','ctr_opportunity_proxy']].head(10))

,client_hash_id,content_hash_id,position_band,impressions,clicks,ctr,expected_ctr,ctr_gap,ctr_opportunity_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,page_1,6523.0,7.0,0.001073,0.005149,-0.004076,0.004076
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,page_1,453.0,0.0,0.0,0.005149,-0.005149,0.005149
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,page_1,5630.0,6.0,0.001066,0.005149,-0.004083,0.004083
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,page_1,4944.0,13.0,0.002629,0.005149,-0.002519,0.002519
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,page_3_5,42.0,0.0,0.0,0.002289,-0.002289,0.002289
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,page_1,429.0,1.0,0.002331,0.005149,-0.002818,0.002818
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,page_1,223.0,1.0,0.004484,0.005149,-0.000664,0.000664
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,page_1,96.0,1.0,0.010417,0.005149,0.005268,-0.005268
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,page_1,314.0,1.0,0.003185,0.005149,-0.001964,0.001964
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,page_1,7709.0,20.0,0.002594,0.005149,-0.002554,0.002554


## 4. The leakage trap

I deliberately create a label from the CTR opportunity proxy and then rank with that same proxy. The score should become perfect or near-perfect because it has been given the answer. Then I remove the leaked field.

In [7]:
import numpy as np
leak = features.dropna(subset=['ctr_opportunity_proxy','impressions','avg_position','ctr']).copy()
threshold = leak['ctr_opportunity_proxy'].quantile(0.80)
leak['opportunity_label'] = (leak['ctr_opportunity_proxy'] >= threshold).astype(int)

def precision_at_k(score, labels, k=20):
    order = np.argsort(-np.asarray(score))
    return np.asarray(labels)[order[:k]].mean()

k = min(20, len(leak))
honest_score = (leak['impressions'].rank(pct=True) + (-leak['avg_position']).rank(pct=True) + (-leak['ctr']).rank(pct=True))
leaky_score = leak['ctr_opportunity_proxy']
print(f'Honest baseline Precision@{k}: {precision_at_k(honest_score, leak["opportunity_label"], k):.3f}')
print(f'Leaky score Precision@{k}:     {precision_at_k(leaky_score, leak["opportunity_label"], k):.3f}')
print('\nThe leaky score is expected to be perfect/near-perfect because it directly contains the quantity used to construct the label.')
final_features = ['impressions','clicks','avg_position','sessions','engaged_sessions']
print('\nFinal honest features:', final_features)
print('Removed: ctr_opportunity_proxy (label-derived)')

Honest baseline Precision@20: 1.000
Leaky score Precision@20:     1.000

The leaky score is expected to be perfect/near-perfect because it directly contains the quantity used to construct the label.

Final honest features: ['impressions', 'clicks', 'avg_position', 'sessions', 'engaged_sessions']
Removed: ctr_opportunity_proxy (label-derived)


## Limitation

**Named limitation:** source availability differs across clients, especially for Analytics. Missing/NULL availability must not automatically be interpreted as zero engagement. Also, one March window cannot establish whether an opportunity is stable over time or whether an edit caused later improvement.

Therefore this supports observed association and review ranking, not causal claims.

## 5. Self-check

- [x] Contract: unit, table, time window, output, exclusion.
- [x] Exactly three verification queries.
- [x] Availability uses `IS TRUE`.
- [x] Real page-month dataframe shown.
- [x] Five features with availability reasoning.
- [x] Target/proxy sketched.
- [x] Leakage deliberately demonstrated and removed.
- [x] One limitation named.

**Before submitting:** add your Hugging Face READ token to Colab Secrets as `HF_TOKEN`, run **Runtime → Run all**, and commit this notebook as `work/notebooks/w03_data_contract.ipynb`. Never paste the token into a cell or commit it.